<h1>Missclassification quickstart notebook</h1>

In this script, we are demonstrating how to initialize and run the EM algorithm based on _Van Delden, A., Scholtus, S., Peerlings, D., Su, L.N., & Windmeijer, D. (2025). Identifying misclassified labels in register data_. 

We will briefly show the prerequisites, the setup, data imports, initialization and fitting of the algorithm, and computation of performance metrics.

## Prequisites ##

### Installation 

This notebook requires all package dependencies listed in `requirements.txt` are installed in the current Python environment. If that is not the case, install them in an environment with Python version >= 3.12 using the Python cell below

In [22]:
# !pip install -r requirements.txt

## Setup ##

### Import packages

In [ ]:
import pandas as pd
import numpy as np
import pickle
import toml
import sys, getopt
import datetime
import logging
import re
from utils.mainV3 import LabelQuality
from utils.utils import get_errorper, select_model, lq_log
from utils.sample import create_sample
from utils.create_input_files import add_selection_columns

### Set paths

In [ ]:
current_path = '/data/projecten/textmining_sbi/sbikwaliteit/LabelQuality/V3/'
pathR = current_path + 'results'
# pathData = current_path + 'data'
pathLog = current_path + "log/"

### Toml configuration file
Read the configuration file and set parameters with run number

In [25]:
config = toml.load(f'{current_path}/config/dry_beans.toml')
# config = toml.load(f'{current_path}/config/config_dry_beans.toml')
models = toml.load(current_path + 'config/models.toml')

# Select model, results filename and run number
base_clf, calibrate, clf_params = select_model('NB', models)
log = 'dry_beans'
run_no = 1

config['base_clf'] = base_clf
config['calibrate'] = calibrate
config['log'] = log

config['prior'] = False
config['r2_grenswaarde'] = 0.1

setseed = config['seeds'][run_no - 1]
run = f"Seed{run_no}"

## Activation

### Loading data

In [26]:
# Load features
with open(f'{current_path}/dry_beans/features.pickle', 'rb') as f:
    X = pickle.load(f)

# Load labels
yplus = pd.read_csv(f'{current_path}/dry_beans/labels_1_test.csv',  sep=';') # Separator in sectie R moet naar ; (is nu ,), in simulaties al ;
yplus['gecontroleerd'] = 'Ja'

#### Parameter logging

In [27]:
logging.basicConfig(filename=f'{pathLog}{log}.log', level=logging.DEBUG, filemode='w')
lq_log(f"run={run}, seed={setseed}")
lq_log(config)
# lq_log(f'Classifier: {clf_name}')
lq_log(f'Parameters: {clf_params}')
lq_log(f'Calibrate: {calibrate}')

run=Seed1, seed=14710
{'trueLabel_col': 'Class', 'performance_col': 'background', 'gold_cols': ['gold100'], 'bg_col': 'background', 'sc_lim': 'background', 'seeds': [14710, 99014, 30681, 25378, 17370, 81638, 78529, 87195, 14258, 84308], 'observedLabel': 'Class_observed', 'errortypes': [''], 'errorper': [''], 'controle': 'gecontroleerd', 'scenarios': ['sc1'], 'tau_choice': 'psi', 'wmodel': 'w4b', 'prior': False, 'include_odf': False, 'base_clf': MultinomialNB(), 'calibrate': True, 'log': 'dry_beans', 'r2_grenswaarde': 0.1}
Parameters: {}
Calibrate: True


### Configuration settings

In [28]:
clf_name = 'NB'

alpha = config.get('alpha', 1)
tau_choice = config.get('tau_choice', 'psi')
errortypes = config['errortypes']
errorper = config['errorper']
observedLabel = config['observedLabel']

threshold = config.get('threshold', 5)
wmodel = config.get('wmodel')
prior_ = config.get('prior')
incl_odf = config.get('include_odf')
r2_grens = config.get('r2_grenswaarde')
it_odf = config.get('it_overduidelijkefout')
X1 = config.get('bg_col', 'X1')
sc_lim = config.get('sc_lim', 'sc_lim')

#### Select Scenario ####

In [29]:
sce = config['scenarios'][0]
# For a given scenario, select the correct values
if sce == 'sc1':
    config['bg_col'] = None
    config['sample_col'] = None
    config['n_sample'] = None
elif sce == 'sc2':
    config['bg_col'] = X1
    config['sample_col'] = None
    config['n_sample'] = None
elif sce == 'sc3':
    config['bg_col'] = sc_lim
    config['sample_col'] = f'sample_{sc_lim}'
elif sce == 'sc4':
    config['bg_col'] = X1
    config['sample_col'] = f'sample_{X1}'

In [ ]:
# Set correct column names for gold, error and percentage strings
gold_col = config['gold_cols'][0]
config['gold_col'] = gold_col

# TODO: opmerking dat het niet aangepast moet worden
err = errortypes[0]
config['errortype'] = err
errorper = get_errorper(config, err)
per = errorper[0]
config['percentage'] = per 

# Log the column name
lq_log(f'scenario: {sce}, gold: {gold_col}, errortype: {err}, percentage: {per}')
config['observedLabel_col'] = f'{observedLabel}{per}{err}'
lq_log(f'{observedLabel}{per}{err}')

scenario: sc1, gold: gold100, errortype: , percentage: 
Class_observed


In [ ]:
# If eps, then load in correct transition matrix files
# if tau_choice == 'eps':
#     errortype = err if per == '' else ''
#     pattern = f"{'' if errortype=='' else errortype + '_'}{threshold}"
#     config['GHTables'] = {
#         'Gh': f"data/GhTable_simulation_{pattern}.csv",
#         'Hg': f"data/HgTable_simulation_{pattern}.csv",
#         'OGh': f"data/OGhTable_simulation_{pattern}.csv",
#         'OHg': f"data/OHgTable_simulation_{pattern}.csv"
#     }

In [32]:
# Initialization of labelquality object
start_time = datetime.datetime.now().replace(microsecond=0)
lq = LabelQuality(X, yplus, config)

          Class  background  gold100 Class_observed Class_observed_bg  \
0         SEKER           2        0       BARBUNYA             SEKER   
1         SEKER           1        0          SEKER             SEKER   
2         SEKER           2        0          SEKER             SEKER   
3         SEKER           3        0          SEKER             SEKER   
4         SEKER           3        0          SEKER             SEKER   
...         ...         ...      ...            ...               ...   
13606  DERMASON           1        0       DERMASON          DERMASON   
13607  DERMASON           2        0       DERMASON          DERMASON   
13608  DERMASON           2        0       DERMASON          DERMASON   
13609  DERMASON           2        0       DERMASON          DERMASON   
13610  DERMASON           2        0       DERMASON          DERMASON   

      gecontroleerd  
0                Ja  
1                Ja  
2                Ja  
3                Ja  
4            

### Initialize initial tau list ###

We intialize a list for initial tau values, each to be iterated over in the EM algorithm for a small number of iterations. After, we select the top three best performing tau values to optimize over for the full number of iterations. 

In [33]:
tau_init_list_sc1 = pd.DataFrame()

# If no existing file, then call initial_tau_multiple, else read from file
tau_init_files = config.get('tau_init', {})
if len(tau_init_files) == 0:
    tau_init_list = lq.initial_tau_multiple(setseed)
    tau_init_list = pd.concat([tau_init_list, tau_init_list_sc1], axis=1, sort=False)
    tau_init_list.columns = range(tau_init_list.shape[1])

else:
    tau_init_list = pd.DataFrame()
    for i, key in enumerate(tau_init_files.keys()):
        results_df = pd.read_csv(tau_init_files[key], sep=',')
        tau_init = results_df[['pred_tau']]
        tau_init.columns = [key]
        tau_init_list = pd.concat([tau_init_list, tau_init], axis=1)


### EM algorithmn ###

In [34]:
list_bestLLH = []

lq_log('processing initial tau_init')

for l in tau_init_list.columns:
    lq_log(f'\ttau_init case: {l}', to_console=False)
    tau_init = tau_init_list[l]
    # for the audit sample & gold set the initial tau to the true value.
    bestLLHs = lq.EM(tau_init=tau_init, compute_results=False, sce = sce)
    list_bestLLH.append(bestLLHs)

n_tau_init = min(len(tau_init_list.columns), 3)
LLHs = pd.DataFrame({"LLH":list_bestLLH, "it":tau_init_list.columns}) 
# save LLH's and which startoption they belong to
bestit = LLHs.sort_values(by="LLH", axis=0, ascending=False)['it'][:n_tau_init] 

# select three best taus
tau_init_df = tau_init_list[bestit]
lq_log('processing best tau_init (max 3)')

processing initial tau_init
y_pi: 0        0.021230
1        0.021231
2        0.021237
3        0.021247
4        0.021995
           ...   
13606    0.035901
13607    0.053496
13608    0.053749
13609    0.046021
13610    0.033313
Length: 13611, dtype: float64 

y_pi_adj: 0        0.021230
1        0.021231
2        0.021237
3        0.021247
4        0.021995
           ...   
13606    0.035901
13607    0.053496
13608    0.053749
13609    0.046021
13610    0.033313
Length: 13611, dtype: float64 

pi_stats =             manipulated pi_est_old    pi_est   afkap_z         z
background                                                      
1              0.284071        NaN  0.038866  0.308019  0.089686
2              0.287513        NaN  0.038866  0.308019  0.085303
3              0.285278        NaN  0.038866  0.308019  0.081236
4              0.276025        NaN  0.038866  0.308019  0.091483
it = 2, a = 2.0
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
ba

In [35]:
# For the top three taus, run full iterations
summary, results_df, pi_stats, df, psi_output, confusion = [None] * n_tau_init, [None] * n_tau_init, [None] * n_tau_init, [None] * n_tau_init, [None] * n_tau_init, [None] * n_tau_init
for i in range(n_tau_init):
    lq_log(f'\tbest tau_init case: {i}', to_console=False)
    print(tau_init_df.iloc[:,i])
    tau_init = tau_init_df.iloc[:,i]
    list_temp =lq.EM(tau_init=tau_init, compute_results=True, sce = sce)
    
    summary[i], results_df[i], pi_stats[i], psi_output[i], confusion[i] = list_temp

    df[i] = float(summary[i].iloc[-1,:][["LLH"]].iloc[0])

#S ave best result
i_max = np.argmax(df)        

0        0.184619
1        0.184617
2        0.184628
3        0.184637
4        0.187724
           ...   
13606    0.217421
13607    0.300810
13608    0.302782
13609    0.269691
13610    0.203371
Name: 14, Length: 13611, dtype: float64
y_pi: 0        0.184619
1        0.184617
2        0.184628
3        0.184637
4        0.187724
           ...   
13606    0.217421
13607    0.300810
13608    0.302782
13609    0.269691
13610    0.203371
Name: 14, Length: 13611, dtype: float64 

y_pi_adj: 0        0.184619
1        0.184617
2        0.184628
3        0.184637
4        0.187724
           ...   
13606    0.217421
13607    0.300810
13608    0.302782
13609    0.269691
13610    0.203371
Length: 13611, dtype: float64 

pi_stats =             manipulated pi_est_old    pi_est   afkap_z         z
background                                                      
1              0.284071        NaN  0.234566  0.558586  0.248002
2              0.287513        NaN  0.234566  0.558586  0.262076
3    

/home/data/projecten/textmining_sbi/sbikwaliteit/LabelQuality/V3/utils/mainV3.py:1345: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, pd.DataFrame.from_records([{


it = 2, a = 2.0
it = 2, a = 1.0
it = 2, a = 0.5
it = 2, a = 0.25
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
background                                                       
1              0.284071    0.234566  0.257428  0.569361  0.257750
2              0.287513    0.234566  0.257428  0.569361  0.270555
3              0.285278    0.234566  0.257428  0.569361  0.256674
4              0.276025    0.234566  0.257428  0.569361  0.257098


/home/data/projecten/textmining_sbi/sbikwaliteit/LabelQuality/V3/utils/mainV3.py:1345: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, pd.DataFrame.from_records([{


it = 3, a = 2.0
it = 3, a = 1.0
it = 3, a = 0.5
it = 3, a = 0.25
it = 3, a = 0.125
it = 3, a = 0.0625
it = 3, a = 0.03125
it = 3, a = 0.015625
pi_stats =             manipulated  pi_est_old   pi_est   afkap_z         z
background                                                      
1              0.284071    0.257428  0.26643  0.571896  0.261454
2              0.287513    0.257428  0.26643  0.571896  0.273895
3              0.285278    0.257428  0.26643  0.571896  0.261632
4              0.276025    0.257428  0.26643  0.571896  0.264196
it = 4, a = 2.0
it = 4, a = 1.0
it = 4, a = 0.5
it = 4, a = 0.25
it = 4, a = 0.125
it = 4, a = 0.0625
it = 4, a = 0.03125
it = 4, a = 0.015625
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
background                                                       
1              0.284071     0.26643  0.270307  0.572151  0.263404
2              0.287513     0.26643  0.270307  0.572151  0.275694
3              0.285278     0.26643  0

...

it = 3, a = 2.0
it = 3, a = 1.0
it = 3, a = 0.5
it = 3, a = 0.25
it = 3, a = 0.125
it = 3, a = 0.0625
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
background                                                       
1              0.284071    0.286294  0.278355  0.575756  0.266134
2              0.287513    0.286294  0.278355  0.575756  0.279548
3              0.285278    0.286294  0.278355  0.575756  0.266590
4              0.276025    0.286294  0.278355  0.575756  0.267350
it = 4, a = 2.0
it = 4, a = 1.0
it = 4, a = 0.5
it = 4, a = 0.25
it = 4, a = 0.125
it = 4, a = 0.0625
it = 4, a = 0.03125
it = 4, a = 0.015625
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
background                                                       
1              0.284071    0.278355  0.274902  0.574484  0.265549
2              0.287513    0.278355  0.274902  0.574484  0.276978
3              0.285278    0.278355  0.274902  0.574484  0.264302
4      

/home/data/projecten/textmining_sbi/sbikwaliteit/LabelQuality/V3/utils/mainV3.py:1345: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, pd.DataFrame.from_records([{


it = 2, a = 2.0
it = 2, a = 1.0
it = 2, a = 0.5
it = 2, a = 0.25
it = 2, a = 0.125
it = 2, a = 0.0625
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
background                                                       
1              0.284071    0.267468  0.271795  0.575446  0.263599
2              0.287513    0.267468  0.271795  0.575446  0.277492
3              0.285278    0.267468  0.271795  0.575446  0.265828
4              0.276025    0.267468  0.271795  0.575446  0.268139


/home/data/projecten/textmining_sbi/sbikwaliteit/LabelQuality/V3/utils/mainV3.py:1345: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, pd.DataFrame.from_records([{


it = 3, a = 2.0
it = 3, a = 1.0
it = 3, a = 0.5
it = 3, a = 0.25
it = 3, a = 0.125
it = 3, a = 0.0625
it = 3, a = 0.03125
it = 3, a = 0.015625
pi_stats =             manipulated  pi_est_old    pi_est   afkap_z         z
background                                                       
1              0.284071    0.271795  0.273277  0.575385  0.264184
2              0.287513    0.271795  0.273277  0.575385  0.278006
3              0.285278    0.271795  0.273277  0.575385  0.266972
4              0.276025    0.271795  0.273277  0.575385  0.268927
it = 4, a = 2.0
it = 4, a = 1.0
it = 4, a = 0.5
it = 4, a = 0.25
it = 4, a = 0.125
it = 4, a = 0.0625
pi_stats =             manipulated  pi_est_old  pi_est   afkap_z         z
background                                                     
1              0.284071    0.273277  0.2739  0.576308  0.264379
2              0.287513    0.273277  0.2739  0.576308  0.277749
3              0.285278    0.273277  0.2739  0.576308  0.266972
4              0.

### Write out metrics ###

The labelquality object writes out several metrics. The summary file covers the standard metrics per iteration, results_df writes out the full data file with all characteristics, pi_stats writes out the corresponding probabilities and fractions of predicted classes, while the confusion folder contains the confusion matrix of the predicted errors

The psi_output contains the psi probability matrix (only applicable if tau_choice is psi)

In [36]:
per_str = '' if per == '' else f'{str(100 + per)[1:]}perc_'
file_name = f'{per_str}{err}_{gold_col.capitalize()}_{sce}_{tau_choice}_{alpha}_{clf_name}_{run}_{wmodel}_prior{prior_}_{config["log"]}_odf{incl_odf}_{r2_grens}_{it_odf}.csv'
summary[i_max].to_csv(f'{pathR}/Summary/{file_name}', index=False)
results_df[i_max].to_csv(f'{pathR}/results_df/{file_name}', index=False)
pi_stats[i_max].to_csv(f'{pathR}/pi_stats/{file_name}', index=False)
confusion[i_max].to_csv(f'{pathR}/confusion/{file_name}', index=False)

if tau_choice == 'psi':
    psi_output[i_max].to_csv(f'{pathR}/psi/{file_name}', index=True)


In [37]:
# Log time 
end_time = datetime.datetime.now().replace(microsecond=0)

lq_log(f'start: {start_time}')
lq_log(f'stop: {end_time}')
lq_log(f'time elapsed: {end_time - start_time}')

start: 2026-01-05 09:38:37
stop: 2026-01-05 09:52:09
time elapsed: 0:13:32


### Results ###

In [38]:
print(confusion[i_max])

Predicted       pred_fout  pred_geenfout
True                                    
echte_fout           2655           1019
echte_geenfout        812           8425


In [39]:
print(summary[i_max].iloc[-1])

Accuracy                 0.858183
Balanced_accuracy        0.817369
TPR                      0.722646
TNR                      0.912093
TPR_sample                   None
TNR_sample                   None
R2                       0.505272
R2_y_gold                0.587837
relEnt_y_noisy           0.574907
LLH_NOISY           -14792.529241
LLH_LAB_PI                    0.0
LLH_LAB_ML             -561.42301
LLH_LAB                -561.42301
LLH                 -15353.952251
switches                      0.0
a                             0.0
Q_pi                 -7512.016086
Q_ml                  -11569.8607
D_r                     -0.002728
Q_ml_old              -11569.8607
Q_pi_old             -7512.013358
Name: 10, dtype: object
